In [44]:
import numpy as np
from numpy.linalg import eig

In [57]:
def hmm(transitions, emissions):
    hidden_states = list(transitions.keys())
    observed_states = list(next(iter(emissions.values())).keys())
    
    t_matrix = np.array([list(state.values()) for state in transitions.values()])
    e_matrix = np.array([list(state.values()) for state in emissions.values()])

    eigenvalues, eigenvectors = np.linalg.eig(t_matrix.T)
    idx = np.argmin(np.abs(eigenvalues - 1))
    v = eigenvectors[:, idx].real
    s_dist = v / v.sum()

    return {
        "T": t_matrix,
        "E": e_matrix,
        "pi": s_dist,
        "states": hidden_states,
        "observations": observed_states
    }

In [58]:
def forward_algorithm(model, observed_seq):
    T = model["T"]
    E = model["E"]
    pi = model["pi"]
    states = model["states"]
    observations = model["observations"]

    n_states = len(states)
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]

    #already computed for first time step: when i = 0
    alphas_old = [ pi[j] * E[j][obs_indices[0]] for j in range(n_states)]

    #computing for other time steps
    for i in range(1, len(observed_seq)):
        alphas_new = [0.0]*n_states
        obs_idx = obs_indices[i]

        #calculating alpha_time_i_(state j) =
        #sum of (alpha_time_i-1 for all states * probability of going from that state to state j )  * emission probability of observance with state j
        for j in range(n_states):
            alphas_new[j] = sum(alphas_old[k]* T[k][j] for k in range(n_states)) * E[j][obs_idx]
        alphas_old = alphas_new

    print(sum(alphas_old))

In [60]:
transitions ={
    "sunny": {"sunny": 0.6, "rainy": 0.3, "cloudy": 0.1},
    "rainy": {"sunny": 0.1, "rainy": 0.5, "cloudy": 0.4},
    "cloudy": {"sunny": 0.3, "rainy": 0.4, "cloudy": 0.3},
}
emissions = {
    "sunny": {"happy": 0.8, "sad": 0.1, "neutral": 0.1},
    "rainy": {"happy": 0.2, "sad": 0.6, "neutral": 0.2},
    "cloudy": {"happy": 0.4, "sad": 0.4, "neutral": 0.2},
}
model_1 = hmm(transitions, emissions)


observed_seq = ["happy", "sad", "happy", "neutral", "neutral"]
forward_algorithm(model_1, observed_seq)

0.0016673602622950825
